In [1]:
# Import standard data manipulation libraries
import pandas as pd
import numpy as np

# Load orders and customers datasets, ensuring date columns are parsed as datetime objects
orders = pd.read_csv("orders.csv", parse_dates=["order_date"])
customers = pd.read_csv("customers.csv", parse_dates=["signup_date"])

# Create a new 'revenue' column by multiplying the price of an item by the quantity bought
orders["revenue"] = orders["price"] * orders["quantity"]

# Define a reference 'snapshot' date (simulating 'today') as one day after the latest order in the dataset to avoid negative recency
snapshot = orders["order_date"].max() + pd.Timedelta(days=1)

# Group the data by customer to calculate Recency, Frequency, and Monetary (RFM) metrics
rfm = orders.groupby("customer_id").agg(
    # Recency: Calculate the number of days between the snapshot date and the customer's most recent order
    recency=("order_date", lambda d: (snapshot - d.max()).days),
    
    # Frequency: Count the number of unique orders placed by the customer
    frequency=("order_id", "nunique"),
    
    # Monetary: Sum the total revenue generated by the customer across all orders
    monetary=("revenue", "sum"),
).reset_index() # Reset the index to convert 'customer_id' back into a standard column

# Display the first 5 rows of the calculated RFM dataframe
rfm.head()

,customer_id,recency,frequency,monetary
0,CUST001,4,3,1893.0
1,CUST002,5,12,19561.0
2,CUST003,7,8,22474.0
3,CUST004,90,2,4373.0
4,CUST006,37,5,17170.0


In [2]:
# Merge the existing RFM data with static customer attributes from the 'customers' table.
# A 'left' join ensures we keep all purchasing customers even if their profile data is missing.
rfm = rfm.merge(customers[["customer_id", "segment", "signup_date", "city"]],
                on="customer_id", how="left")

# Calculate how long the customer has been with the store (tenure) in days.
# The .dt.days accessor extracts the integer number of days from the timedelta object.
rfm["tenure_days"] = (snapshot - rfm["signup_date"]).dt.days

# Create a new behavioral feature for Average Order Value (AOV) by dividing total spend by total orders.
rfm["avg_order_value"] = rfm["monetary"] / rfm["frequency"]

# Select and reorder the final set of numeric and categorical columns to create the clean feature dataset.
feat = rfm[["customer_id", "recency", "frequency", "monetary",
            "avg_order_value", "tenure_days", "segment", "city"]]

# Display the first 5 rows of the finalized feature table.
feat.head()

,customer_id,recency,frequency,monetary,avg_order_value,tenure_days,segment,city
0,CUST001,4,3,1893.0,631.000000,344,Regular,Hyderabad
1,CUST002,5,12,19561.0,1630.083333,726,Regular,Delhi
2,CUST003,7,8,22474.0,2809.250000,640,Regular,Ahmedabad
3,CUST004,90,2,4373.0,2186.500000,654,NaN,Mumbai
4,CUST006,37,5,17170.0,3434.000000,300,Regular,Chennai


In [4]:
# Apply one-hot encoding to convert categorical columns ('segment', 'city') into numerical boolean/binary columns.
# drop_first=True avoids the dummy variable trap (multicollinearity) by dropping the first category alphabetically, making it the hidden baseline.
encoded = pd.get_dummies(feat, columns=["segment", "city"], drop_first=True)

# Isolate and view just the newly created columns related to "segment" (e.g., segment_Regular) to verify the encoding.
# .head(3) displays the first three rows of this filtered view.
encoded.filter(like="segment").head(3)


,segment_Regular
0,True
1,True
2,True


In [5]:

# Extract and output all current column names as a standard Python list.
# This helps you inspect the exact names of all the newly generated dummy columns (like 'city_Mumbai', 'city_Delhi', etc.).
encoded.columns.tolist()

['customer_id',
 'recency',
 'frequency',
 'monetary',
 'avg_order_value',
 'tenure_days',
 'segment_Regular',
 'city_Bengaluru',
 'city_Chennai',
 'city_Delhi',
 'city_Hyderabad',
 'city_Kolkata',
 'city_Mumbai',
 'city_Pune']

In [6]:
# Import the StandardScaler, which standardizes features by removing the mean and scaling to unit variance.
from sklearn.preprocessing import StandardScaler

# Define the list of continuous numeric columns that require scaling so they share a common scale.
num_cols = ["recency", "frequency", "monetary", "avg_order_value", "tenure_days"]

# Initialize the scaler object.
scaler = StandardScaler()

# Compute the mean and standard deviation for the specified columns (fit), 
# apply the transformation (transform), and overwrite the original columns with the scaled values.
encoded[num_cols] = scaler.fit_transform(encoded[num_cols])

# Verify the transformation was successful by viewing summary statistics for the scaled columns.
# We round to 2 decimal places and select specific rows to confirm the mean is ~0 and the standard deviation is 1.
encoded[num_cols].describe().round(2).loc[["mean", "std", "min", "max"]]

,recency,frequency,monetary,avg_order_value,tenure_days
mean,-0.00,-0.00,-0.00,-0.00,0.0
std,1.00,1.00,1.00,1.00,1.0
min,-0.96,-1.15,-1.00,-1.49,-1.8
max,3.07,4.44,5.25,10.30,1.7


In [10]:
# Import the function needed to split the dataset into training and testing subsets
from sklearn.model_selection import train_test_split

# Create the feature matrix (X) by dropping the 'customer_id' column.
# Identifiers hold no predictive value and should not be fed into a machine learning model.
X = encoded.drop(columns=["customer_id"])

# (label y comes in the churn lesson; here we just demonstrate the split)

# Split the data into training and testing sets.
# test_size=0.2 reserves 20% of the data for testing and 80% for training.
# random_state=42 ensures the random shuffle is reproducible every time you run the code.
X_train, X_test = train_test_split(X, test_size=0.2, random_state=42)

# Print the dimensions (rows, columns) of the new datasets to verify the split was successful.
print("train:", X_train.shape, " test:", X_test.shape)

train: (1056, 13)  test: (264, 13)
